# Tetrax OCR — Fine-Tuning on Google Colab
### Dataset: [`sarvamai/indic-ocr-bench`](https://huggingface.co/datasets/sarvamai/indic-ocr-bench)
### Repository: [https://github.com/TusharTemgire/tetrax-OCR-ITT.git](https://github.com/TusharTemgire/tetrax-OCR-ITT.git)

---
This notebook provides a complete, runnable step-by-step workflow to:
1. **Check GPU hardware** in Google Colab (T4 / A100).
2. **Mount Google Drive** to save model checkpoints permanently.
3. **Clone the repository** and install all required CUDA & PyTorch / Paddle libraries.
4. **Fine-tune your custom OCR model** using **Option A (PaddleOCR)** or **Option B (TrOCR Transformer)** directly on the Sarvam Indic OCR dataset.
5. **Benchmark & evaluate** Character Error Rate (CER) and Word Error Rate (WER).
6. **Export static inference weights** ready for production API deployment.

## Step 1: Verify Colab GPU Acceleration
*Make sure you selected **Runtime > Change runtime type > T4 GPU** (or A100).*

In [ ]:
!nvidia-smi

## Step 2: Mount Google Drive
Mounting Google Drive ensures your fine-tuned model checkpoints (`.pdparams` / Hugging Face weights) are saved to your Drive and will not be lost if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create permanent checkpoint folder in your Drive
!mkdir -p /content/drive/MyDrive/tetrax_ocr_models

## Step 3: Clone the Tetrax OCR Repository

In [ ]:
!rm -rf /content/tetrax-ocr
!git clone https://github.com/TusharTemgire/tetrax-OCR-ITT.git /content/tetrax-ocr
%cd /content/tetrax-ocr

## Step 4: Install Dependencies

In [ ]:
# Upgrade pip
!python -m pip install --upgrade pip

# Install PyTorch, Hugging Face Transformers, Datasets, Tokenizers, and Vision Tools
!pip install datasets transformers accelerate jiwer pillow opencv-python-headless pypdfium2 tqdm sentencepiece tiktoken

# Install PaddlePaddle with CUDA GPU support & PaddleOCR
!pip install paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu120/
!pip install "paddleocr>=2.7"

### Verify CUDA Acceleration in Both Engines

In [ ]:
import torch
import paddle

print("--- PYTORCH ---")
print("PyTorch CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

print("\n--- PADDLEPADDLE ---")
print("Paddle CUDA Compiled:", paddle.is_compiled_with_cuda())
if paddle.is_compiled_with_cuda():
    print("Paddle GPU Device:", paddle.device.cuda.get_device_name(0))

## Step 5: Option A — Fine-Tune PaddleOCR on `sarvamai/indic-ocr-bench`
PaddleOCR is the recommended high-throughput engine for production documents.

### 5.1 Extract & Format Dataset from Hugging Face
This pulls from `sarvamai/indic-ocr-bench`, generates PaddleOCR tab-delimited label files, and extracts the Unicode character dictionary (`sarvam_dict.txt`).

In [ ]:
%cd /content/tetrax-ocr

# Extract Hindi and English samples (or change to 'all' for all 22 Indian languages)
!python -m training.load_sarvam_dataset \
  --split small_representative \
  --languages hindi,english \
  --output-dir datasets/sarvam_indic

### 5.2 Clone PaddleOCR Training Framework & Download Pretrained Base Weights

In [ ]:
!git clone https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR
%cd /content/PaddleOCR
!pip install -r requirements.txt

# Download base multilingual PP-OCR training weights
!mkdir -p pretrain_models
%cd pretrain_models
!wget -nc https://paddleocr.bj.bcebos.com/PP-OCRv4/multilingual/multilingual_PP-OCRv4_rec_train.tar
!tar -xf multilingual_PP-OCRv4_rec_train.tar
%cd /content/PaddleOCR

### 5.3 Launch PaddleOCR Fine-Tuning
Checkpoints are saved directly to your mounted Google Drive (`/content/drive/MyDrive/tetrax_ocr_models/paddle_checkpoints`).

In [ ]:
%cd /content/PaddleOCR

!python tools/train.py \
  -c /content/tetrax-ocr/configs/rec_sarvam_indic.yml \
  -o Global.use_gpu=True \
     Global.epoch_num=30 \
     Global.pretrained_model=./pretrain_models/multilingual_PP-OCRv4_rec_train/best_accuracy \
     Global.character_dict_path=/content/tetrax-ocr/datasets/sarvam_indic/dicts/sarvam_dict.txt \
     Global.save_model_dir=/content/drive/MyDrive/tetrax_ocr_models/paddle_checkpoints \
     Train.dataset.data_dir=/content/tetrax-ocr \
     Train.dataset.label_file_list='["/content/tetrax-ocr/datasets/sarvam_indic/paddle_rec/train.txt"]' \
     Eval.dataset.data_dir=/content/tetrax-ocr \
     Eval.dataset.label_file_list='["/content/tetrax-ocr/datasets/sarvam_indic/paddle_rec/val.txt"]'

### 5.4 Export Inference Model for Production Deployment

In [ ]:
%cd /content/PaddleOCR

!python tools/export_model.py \
  -c /content/tetrax-ocr/configs/rec_sarvam_indic.yml \
  -o Global.checkpoints=/content/drive/MyDrive/tetrax_ocr_models/paddle_checkpoints/best_accuracy \
     Global.save_inference_dir=/content/drive/MyDrive/tetrax_ocr_models/paddle_inference

## Step 6: Option B — In-Memory Fine-Tuning with TrOCR (Hugging Face)
Loads `sarvamai/indic-ocr-bench` directly in memory without requiring any local images or CSV files on disk.

In [ ]:
%cd /content/tetrax-ocr

!python -m training.train_trocr \
  --dataset "sarvamai/indic-ocr-bench" \
  --split small_representative \
  --languages hindi,english \
  --output-dir /content/drive/MyDrive/tetrax_ocr_models/trocr_checkpoint \
  --epochs 3 \
  --batch-size 8 \
  --lr 5e-5

## Step 7: Benchmark Accuracy (CER, WER, Exact Match)
Benchmark your model directly across Indian languages from `sarvamai/indic-ocr-bench`.

In [ ]:
%cd /content/tetrax-ocr

!python -m training.benchmark_sarvam \
  --split small_representative \
  --languages hindi,english \
  --max-samples 50

## Step 8: Test Inference on an Example Document

In [ ]:
%cd /content/tetrax-ocr

# Run OCR with PaddleOCR engine
from app.ocr_engine import extract_image
from datasets import load_dataset
import numpy as np
import cv2

# Fetch one test image from Sarvam benchmark
ds = load_dataset("sarvamai/indic-ocr-bench", split="small_representative")
sample = ds[0]
pil_img = sample["image"]
ground_truth = sample.get("gt", "")
lang = sample.get("language", "en")

print(f"Document Language: {lang}")
print(f"Ground Truth:\n{ground_truth}\n")

# Convert to OpenCV BGR for inference
img_bgr = cv2.cvtColor(np.array(pil_img.convert("RGB")), cv2.COLOR_RGB2BGR)
result = extract_image(img_bgr, lang="hi" if "hi" in lang else "en", apply_preprocessing=True)

print("=" * 40)
print("PREDICTED TEXT:")
print(result["text"])
print("=" * 40)